In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_payment_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PaymentId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "PaymentKey	string,"
        #     "PaymentUniqueNumber	string,"
        #     "ExposureKey	string,"
        #     "PaymentActualDate	date,"
        #     "PaymentRecordDate	timestamp,"
        #     "PaymentAmount	decimal(15,2),"
        #     "PaymentIntIdentification	string,"
        #     "PaymentCostAmount	decimal(15,2),"
        #     "CoverageAmount	double,"
        #     "DebtAmount	double,"
        #     "MinimumPayment	double,"
        #     "CurrencyCode	string,"
        #     "Currency	string,"
        #     "PaymentTypeCode	string,"
        #     "PaymentType	string,"
        #     "ReasonCode	string,"
        #     "TransactionReasonCode	string,"
        #     "TransactionReason	string,"
        #     "ReportingDate	string,"
        #     "PrincipalAmount	decimal(15,2),"
        #     "CommissionAmount	decimal(15,2),"
        #     "InterestAmount	decimal(15,2),"
        #     "ExpensesAmount	decimal(15,2),"
        #     "InterestURL	string,"
        #     "Createdby	string,"
        #     "IsDownPaymentFlag	boolean,"
        #     "IsReversedFlag	string,"
        #     "PaymentISCOL	string,"
        #     "PaymentRecordType	string,"
        #     "PaymentTypeGroup string,"
        #     "PaymentTypeEng string,"
        #     "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Payment"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== eaisig_trn_pay ==================

        eaisig_trn_pay_extra_filters = None

        if eaisig_trn_pay_extra_filters is not None:
            eaisig_trn_pay_extra_filters.append(date_filter)
            eaisig_trn_pay_final_filters = " AND ".join(eaisig_trn_pay_extra_filters)
        else:
            eaisig_trn_pay_final_filters = date_filter

            eaisig_trn_pay_selected_fields = [
                "LoadID",
                "eaiSIG_TRN_PAY_PAY_S_ID",
                "eaiSIG_TRN_PAY_CASE_S_ID",
                "eaiSIG_TRN_PAY_PAY_DATE",
                "eaiSIG_TRN_PAY_POST_DATE",
                "eaiSIG_TRN_PAY_PAY_AMT",
                "eaiSIG_TRN_PAY_CURR_CODE",
                "eaiSIG_TRN_PAY_PAY_REASON",
                "eaiSIG_TRN_PAY_TRANSACTION_REAS",
                "eaiSIG_TRN_PAY_TRANSACTION_REAS_DESC",
                "eaiSIG_TRN_PAY_REG_DATE",
                "eaiSIG_TRN_PAY_PAY_ISCOL",
                "eaiSIG_TRN_PAY_RECORD_TYPE",
                "eaiSIG_TRN_PAY_PAY_REASON_DESC",
                "businessdate",
            ]

            eaisig_trn_pay = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "arotron_eaisig_trn_pay",
                eaisig_trn_pay_final_filters,
                eaisig_trn_pay_selected_fields,
            )
        # ================== payment_arotron_df ==================
        payment_arotron_primary_key_fields = [
            eaisig_trn_pay.eaiSIG_TRN_PAY_PAY_S_ID,
            lit("arotron"),
        ]
        payment_arotron_df = eaisig_trn_pay.filter(
            (eaisig_trn_pay.eaiSIG_TRN_PAY_PAY_ISCOL == 1)
            & (eaisig_trn_pay.eaiSIG_TRN_PAY_RECORD_TYPE == 2)
        ).withColumn(
            "PaymentId",
            sha2(concat_ws(";", *payment_arotron_primary_key_fields), 256),
        )

        payment_arotron_df = payment_arotron_df.select(
            payment_arotron_df.PaymentId,
            lit("arotron").alias("SourceSystem"),
            lit("-9998").cast("string").alias("PortfolioKey"),
            lit("Core Wholesale").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            payment_arotron_df.eaiSIG_TRN_PAY_PAY_S_ID.cast("string").alias(
                "PaymentKey"
            ),
            lit(None).alias("PaymentUniqueNumber"),
            payment_arotron_df.eaiSIG_TRN_PAY_CASE_S_ID.alias("ExposureKey"),
            payment_arotron_df.eaiSIG_TRN_PAY_PAY_DATE.cast("date").alias(
                "PaymentActualDate"
            ),
            payment_arotron_df.eaiSIG_TRN_PAY_POST_DATE.cast("timestamp").alias(
                "PaymentRecordDate"
            ),
            payment_arotron_df.eaiSIG_TRN_PAY_PAY_AMT.cast("decimal(15,2)").alias(
                "PaymentAmount"
            ),
            lit(None).alias("PaymentIntIdentification"),
            lit(None).alias("PaymentCostAmount"),
            lit(None).alias("CoverageAmount"),
            lit(None).alias("DebtAmount"),
            lit(None).alias("MinimumPayment"),
            payment_arotron_df.eaiSIG_TRN_PAY_CURR_CODE.alias("CurrencyCode"),
            payment_arotron_df.eaiSIG_TRN_PAY_CURR_CODE.alias("Currency"),
            payment_arotron_df.eaiSIG_TRN_PAY_PAY_REASON.alias("PaymentTypeCode"),
            payment_arotron_df.eaiSIG_TRN_PAY_PAY_REASON_DESC.cast("string").alias("PaymentType"),
            lit(None).alias("ReasonCode"),
            payment_arotron_df.eaiSIG_TRN_PAY_TRANSACTION_REAS.alias(
                "TransactionReasonCode"
            ),
            payment_arotron_df.eaiSIG_TRN_PAY_TRANSACTION_REAS_DESC.cast(
                "string"
            ).alias("TransactionReason"),
            payment_arotron_df.eaiSIG_TRN_PAY_REG_DATE.alias("ReportingDate"),
            lit(None).alias("PrincipalAmount"),
            lit(None).alias("CommissionAmount"),
            lit(None).alias("InterestAmount"),
            lit(None).alias("ExpensesAmount"),
            lit(None).alias("InterestURL"),
            lit(None).alias("Createdby"),
            lit(None).alias("IsDownPaymentFlag"),
            lit(None).alias("IsReversedFlag"),
            payment_arotron_df.eaiSIG_TRN_PAY_PAY_ISCOL.alias("PaymentISCOL"),
            payment_arotron_df.eaiSIG_TRN_PAY_RECORD_TYPE.alias("PaymentRecordType"),
            lit(None).alias("PaymentTypeGroup"),
            lit(None).alias("PaymentTypeEng"),
            payment_arotron_df.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        payment_arotron_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_payment_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)